# Module 3: Clone, Modify, Deploy

Clone the Echo environment from the OpenEnv repo, modify it, test locally, and deploy to HF Spaces.

**Time:** ~25 min · **Difficulty:** Intermediate · **GPU:** Not required

> **Note:** Deployment to HF Spaces (Step 6) requires a Hugging Face account and token.
> All other steps run locally.

In [ ]:
# Install dependencies and set up the OpenEnv repository
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openenv-core', 'fastmcp', 'fastapi', 'uvicorn'], check=True)

if not os.path.exists('OpenEnv'):
    subprocess.run(['git', 'clone', 'https://github.com/meta-pytorch/OpenEnv.git',
                    '--depth=1', '-q'], check=True)

repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Setup complete!')

## 1. Verify the Hosted Echo Environment

First, let's confirm the hosted Echo environment works.

In [ ]:
from envs.echo_env import EchoEnv

ECHO_URL = 'https://openenv-echo-env.hf.space'

with EchoEnv(base_url=ECHO_URL).sync() as env:
    result = env.reset()
    response = env.call_tool('echo_message', message='ping')
    print(f'Sent: ping')
    print(f'Received: {response}')
    print('The standard Echo returns exactly what you send.')

## 2. Clone the Echo Environment

Clone the Space repository to get the full source code.

In [ ]:
# Copy the echo_env from the cloned OpenEnv repo into a working directory
import shutil

src = os.path.join(os.path.abspath('OpenEnv'), 'envs', 'echo_env')
dst = 'echo-env-modified'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print('Copied echo_env to echo-env-modified/')
os.listdir(dst)

## 3. Explore the Structure

Every OpenEnv environment follows the same layout.

In [ ]:
import glob
files = sorted(glob.glob('echo-env-modified/**/*', recursive=True))
for f in files:
    if os.path.isfile(f):
        print(f)

In [ ]:
# Look at the MCP tool definitions in the echo environment
env_file = 'echo-env-modified/server/echo_environment.py'
with open(env_file) as f:
    print(f.read())

## 4. Modify the Environment

Let's make a "Reverse Echo" — instead of echoing back the message, it reverses it.

We'll modify the `step()` method in `environment.py`.

In [ ]:
env_file = 'echo-env-modified/server/echo_environment.py'

with open(env_file) as f:
    content = f.read()

print('Original echo_environment.py:')
print(content)

In [ ]:
# Modify: make echo_message reverse the input
# The MCP tool currently returns `message`; we change it to `message[::-1]`

modified = content.replace(
    'return message',
    'return message[::-1]',
    1  # Replace only the first occurrence (in echo_message tool)
)

with open(env_file, 'w') as f:
    f.write(modified)

print('Modified echo_environment.py (echo_message now reverses the input):')
# Show the relevant section
for line in modified.split('\n'):
    if 'echo_message' in line or 'return' in line or '@mcp' in line:
        print(f'  {line}')

## 5. Test Locally

Start the modified server and connect to it.

> In Colab, we'll start the server as a background process. Locally, you'd run `uv run server` in a separate terminal.

In [ ]:
import subprocess
import time
import sys
import os

# The server app imports from openenv (installed) and envs.echo_env (in OpenEnv repo).
# We run from the echo-env-modified directory so its server/ is importable.
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([
    os.path.abspath('echo-env-modified'),
    os.path.abspath('OpenEnv'),
    os.path.abspath('OpenEnv/src'),
] + env.get('PYTHONPATH', '').split(os.pathsep))

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8001'],
    cwd='echo-env-modified',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give it time to start
time.sleep(4)
print(f'Server started (PID: {server.pid})')

# Check it's healthy
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8001/health', timeout=5) as r:
        print(f'Health: {r.read().decode()}')
except Exception as e:
    print(f'Health check failed: {e}')
    # Print server stderr for debugging
    err = server.stderr.read1(1024).decode(errors='replace')
    if err:
        print(f'Server stderr: {err}')

In [ ]:
# Test the modified environment
# Since this is an MCP env, we use EchoEnv.call_tool()
from envs.echo_env import EchoEnv

with EchoEnv(base_url='http://localhost:8001').sync() as env:
    result = env.reset()

    test_messages = ['Hello', 'OpenEnv', 'Reverse this!']
    for msg in test_messages:
        response = env.call_tool('echo_message', message=msg)
        print(f'Sent: {msg:20s} -> Received: {response}')

In [ ]:
# Clean up the server
server.terminate()
server.wait()
print("Server stopped.")

## 6. Deploy to HF Spaces

Once your environment works locally, deploy it with `openenv push`.

```bash
cd echo-env-modified
openenv push --repo-id YOUR_USERNAME/reverse-echo-env
```

Your environment is now live at:
- **API:** `https://YOUR_USERNAME-reverse-echo-env.hf.space`
- **Web UI:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/web`
- **Docs:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/docs`

In [ ]:
# Uncomment and run to deploy (requires HF token)
# !cd echo-env-modified && openenv push --repo-id YOUR_USERNAME/reverse-echo-env

## 7. Connect to Your Deployed Environment

After deployment, install the client and connect:

In [ ]:
# Uncomment after deploying
# !pip install -q git+https://huggingface.co/spaces/YOUR_USERNAME/reverse-echo-env
#
# with EchoEnv(base_url="https://YOUR_USERNAME-reverse-echo-env.hf.space").sync() as env:
#     result = env.reset()
#     result = env.step(EchoAction(message="Deployed!"))
#     print(f"Response from your Space: {result.observation}")

## 8. Docker Deployment (Alternative)

You can also pull and run the Docker image locally:

```bash
# Pull from HF registry (after deploying)
docker pull registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest
docker run -d -p 8001:8000 registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest

# Or build from source
cd echo-env-modified
docker build -t reverse-echo:latest -f server/Dockerfile .
docker run -d -p 8001:8000 reverse-echo:latest
```

Connect the same way:
```python
with EchoEnv(base_url="http://localhost:8001").sync() as env:
    result = env.reset()
```

## Summary

What you did:
1. Cloned an existing environment from HF Spaces
2. Explored its structure (models, client, server)
3. Modified the environment logic (echo → reverse echo)
4. Tested locally with uvicorn
5. Deployed to HF Spaces with `openenv push`

The workflow is always: **clone → modify → test → deploy**.

**Next:** [Module 4](../module-4/README.md) — Building an environment from scratch.